# INITIAL CONFIGURATION

In [1]:
import os

print(os.getcwd())
if not os.getcwd().endswith("app"):
    os.chdir("../app")
    print(os.getcwd())

import pandas as pd
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)

# %load_ext autoreload
# %autoreload 2
%matplotlib inline

/home/turbotowerlnx/Documents/Master/SMA/SMA-MultiAgent-Based-Crowd-Simulation/notebooks
/home/turbotowerlnx/Documents/Master/SMA/SMA-MultiAgent-Based-Crowd-Simulation/app


# ANALYSIS

In [2]:
from maikol_utils.file_utils import list_dir_files
from src.config import Configuration

CONFIG = Configuration()

experiments, _ = list_dir_files(CONFIG.LOGS_DIR)
experiments = [exp for exp in experiments if ".csv" in exp]

experiments

['../logs/experiments_results_aggressive_agents_impact.csv',
 '../logs/experiments_results_evaluate_different_agent_densities.csv',
 '../logs/experiments_results_slow_agents_impact.csv']

### Number of agents

In [12]:
df_n_agents = pd.read_csv(experiments[1])


df_n_agents.groupby(["RunId", "iteration", "initial_agents"])["Step"].max()
# df_n_agents.head(200)

RunId  iteration  initial_agents
0      0          10                 19
1      0          10                 70
2      0          25                 16
3      0          25                 72
4      0          50                 28
                                   ... 
19995  999        175               153
19996  999        200                48
19997  999        200               162
19998  999        300                62
19999  999        300               242
Name: Step, Length: 20000, dtype: int64

In [13]:

# Get max steps per run
max_steps_per_run = df_n_agents.groupby(["RunId", "initial_agents"])["Step"].mean().reset_index()
max_steps_per_run.columns = ["RunId", "initial_agents", "max_steps"]
max_steps_per_run.head(10)


,RunId,initial_agents,max_steps
0,0,10,9.5
1,1,10,35.0
2,2,25,8.0
3,3,25,36.0
4,4,50,14.0
5,5,50,34.5
6,6,75,14.5
7,7,75,37.5
8,8,100,20.0
9,9,100,52.5


In [14]:

# Compute statistics per experiment type (initial_agents)
stats_per_experiment = max_steps_per_run.groupby("initial_agents")["max_steps"].agg([
    "count",      # number of runs
    "mean",       # mean max steps
    "median",     # median max steps
    "std",        # standard deviation
    "min",        # minimum max steps
    "max",        # maximum max steps
    ("q25", lambda x: x.quantile(0.25)),  # 25th percentile
    ("q75", lambda x: x.quantile(0.75)),  # 75th percentile
]).round(2)

print("Statistics of Max Steps per Experiment Type:")
print(stats_per_experiment)


Statistics of Max Steps per Experiment Type:
                count   mean  median    std   min    max   q25    q75
initial_agents                                                       
10               2000  18.64   17.00   7.79   4.5   45.0  11.5   25.5
25               2000  21.74   21.50   8.65   8.0   50.0  13.5   29.5
50               2000  24.84   24.25  10.48  10.0   51.5  14.5   34.5
75               2000  28.20   28.50  12.82  11.5   60.5  15.5   40.0
100              2000  31.94   29.50  15.99  11.5   74.0  16.0   47.0
125              2000  36.43   33.00  19.30  12.5   76.5  17.5   55.0
150              2000  41.34   37.00  22.72  15.0   87.0  19.0   63.0
175              2000  46.71   41.25  26.16  17.0   99.5  21.0   72.0
200              2000  52.59   46.75  30.00  18.5  114.0  23.0   81.5
300              2000  75.50   68.25  43.99  27.0  151.5  31.5  119.0


In [15]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Create subplots: box plot and line plot with error bars
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Distribution of Max Steps by Initial Agents", 
                    "Mean Max Steps by Initial Agents (with Std Dev)"),
    specs=[[{}, {}]]
)

# Box plot
for agent_count in sorted(max_steps_per_run["initial_agents"].unique()):
    data = max_steps_per_run[max_steps_per_run["initial_agents"] == agent_count]["max_steps"]
    fig.add_trace(
        go.Box(y=data, name=f"{agent_count}", showlegend=False),
        row=1, col=1
    )

# Line plot with error bars
means = stats_per_experiment["mean"]
stds = stats_per_experiment["std"]
fig.add_trace(
    go.Scatter(
        x=means.index,
        y=means.values,
        error_y=dict(type="data", array=stds.values, visible=True),
        mode="lines+markers",
        name="Mean ± Std",
        marker=dict(size=8),
        line=dict(width=2)
    ),
    row=1, col=2
)

# Update layout
fig.update_xaxes(title_text="Initial Agents", row=1, col=1)
fig.update_yaxes(title_text="Max Steps", row=1, col=1)
fig.update_xaxes(title_text="Initial Agents", row=1, col=2)
fig.update_yaxes(title_text="Max Steps", row=1, col=2)

fig.update_layout(height=500, width=1200, showlegend=True)
fig.show()

### Aggressive impact

In [16]:
df_aggressive = pd.read_csv(experiments[0])
# df_aggressive.head(200)

In [17]:
df_aggressive["scenario_type"].unique()

array(['MALL', 'SEATS'], dtype=object)

In [18]:

# Get max steps per run for aggressive ratio analysis
max_steps_aggressive = df_aggressive.groupby(["RunId", "aggressive_ratio"])["Step"].max().reset_index()
max_steps_aggressive.columns = ["RunId", "aggressive_ratio", "max_steps"]

# Compute statistics per aggressive ratio
stats_aggressive = max_steps_aggressive.groupby("aggressive_ratio")["max_steps"].agg([
    "count",
    "mean",
    "median",
    "std",
    "min",
    "max",
    ("q25", lambda x: x.quantile(0.25)),
    ("q75", lambda x: x.quantile(0.75)),
]).round(2)

print("\n" + "=" * 80)
print("AGGRESSIVE AGENTS IMPACT ANALYSIS")
print("=" * 80)
print("\nStatistics of Max Steps per Aggressive Ratio:")
print(stats_aggressive)



AGGRESSIVE AGENTS IMPACT ANALYSIS

Statistics of Max Steps per Aggressive Ratio:
                  count   mean  median    std  min  max    q25    q75
aggressive_ratio                                                     
0.00               2000  72.73    68.0  38.37   27  152  35.00  109.0
0.10               2000  70.90    68.5  37.11   26  158  34.00  106.0
0.25               2000  68.91    60.0  35.68   25  150  34.00  103.0
0.50               2000  66.35    62.5  34.08   25  151  33.00   99.0
0.80               2000  64.18    64.5  32.68   24  132  32.00   95.0
1.00               2000  63.24    61.0  32.02   25  136  31.75   94.0


In [19]:

# Calculate growth metrics for aggressive ratio
agg_analysis = stats_aggressive.copy()
agg_analysis['coefficient_of_variation'] = (agg_analysis['std'] / agg_analysis['mean']).round(3)
agg_analysis['mean_growth_rate'] = agg_analysis['mean'].pct_change().round(3)

print("\n" + "=" * 80)
print("DETAILED GROWTH ANALYSIS - AGGRESSIVE AGENTS")
print("=" * 80)
print("\nMetrics by Aggressive Ratio:")
print(agg_analysis[['mean', 'std', 'coefficient_of_variation', 'mean_growth_rate']])

# Impact analysis
print("\n" + "=" * 80)
print("IMPACT OF AGGRESSIVE AGENTS:")
print("=" * 80)

baseline = stats_aggressive['mean'].iloc[0]
for idx, ratio in enumerate(stats_aggressive.index):
    mean_steps = stats_aggressive.loc[ratio, 'mean']
    cv = agg_analysis.loc[ratio, 'coefficient_of_variation']
    impact = ((mean_steps - baseline) / baseline * 100) if idx > 0 else 0
    status = "🟢 MINIMAL" if cv < 0.35 else "🟡 MODERATE" if cv < 0.45 else "🔴 HIGH"
    
    print(f"\n{ratio*100:5.0f}% aggressive:")
    print(f"   Mean steps: {mean_steps:7.1f} ({impact:+6.1f}% vs baseline)")
    print(f"   Std dev:    {stats_aggressive.loc[ratio, 'std']:7.1f}")
    print(f"   CV:         {cv:.3f} {status}")



DETAILED GROWTH ANALYSIS - AGGRESSIVE AGENTS

Metrics by Aggressive Ratio:
                   mean    std  coefficient_of_variation  mean_growth_rate
aggressive_ratio                                                          
0.00              72.73  38.37                     0.528               NaN
0.10              70.90  37.11                     0.523            -0.025
0.25              68.91  35.68                     0.518            -0.028
0.50              66.35  34.08                     0.514            -0.037
0.80              64.18  32.68                     0.509            -0.033
1.00              63.24  32.02                     0.506            -0.015

IMPACT OF AGGRESSIVE AGENTS:

    0% aggressive:
   Mean steps:    72.7 (  +0.0% vs baseline)
   Std dev:       38.4
   CV:         0.528 🔴 HIGH

   10% aggressive:
   Mean steps:    70.9 (  -2.5% vs baseline)
   Std dev:       37.1
   CV:         0.523 🔴 HIGH

   25% aggressive:
   Mean steps:    68.9 (  -5.3% vs baselin

In [20]:

# Check unique scenario types first
print("Unique scenario types in data:", df_aggressive['scenario_type'].unique())
print("Unique aggressive ratios:", sorted(df_aggressive['aggressive_ratio'].unique()))

# Get max steps per run including scenario_type
max_steps_agg_by_scenario = df_aggressive.groupby(["RunId", "scenario_type", "aggressive_ratio"])["Step"].max().reset_index()
max_steps_agg_by_scenario.columns = ["RunId", "scenario_type", "aggressive_ratio", "max_steps"]

print("\nSample of data with scenario type:")
print(max_steps_agg_by_scenario.head(20))

# Visualization for aggressive agent impact
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Distribution of Max Steps by Aggressive Ratio", 
                    "Mean Max Steps by Aggressive Ratio - By Scenario Type"),
    specs=[[{}, {}]]
)

# Box plot (overall, not by scenario)
for ratio in sorted(max_steps_aggressive["aggressive_ratio"].unique()):
    data = max_steps_aggressive[max_steps_aggressive["aggressive_ratio"] == ratio]["max_steps"]
    fig.add_trace(
        go.Box(y=data, name=f"{ratio*100:.0f}%", showlegend=False),
        row=1, col=1
    )

# Line plot with error bars - separate traces by scenario_type
scenario_types = sorted(df_aggressive['scenario_type'].unique())
colors = ["blue", "red", "green", "orange", "purple", "brown", "pink", "cyan"]

print(f"\nCreating traces for {len(scenario_types)} scenario types: {scenario_types}")

for i, scenario in enumerate(scenario_types):
    # Filter data for this scenario
    scenario_max_steps = max_steps_agg_by_scenario[max_steps_agg_by_scenario['scenario_type'] == scenario]
    
    # Get stats per aggressive ratio for this scenario
    scenario_stats = scenario_max_steps.groupby("aggressive_ratio")["max_steps"].agg([
        "mean",
        "std",
    ]).reset_index()
    
    print(f"\n{scenario}: {len(scenario_stats)} data points")
    print(scenario_stats)
    
    if len(scenario_stats) > 0:
        fig.add_trace(
            go.Scatter(
                x=[f"{x*100:.0f}%" for x in scenario_stats["aggressive_ratio"]],
                y=scenario_stats["mean"].values,
                error_y=dict(type="data", array=scenario_stats["std"].values, visible=True),
                mode="lines+markers",
                name=scenario,
                marker=dict(size=10),
                line=dict(width=3, color=colors[i % len(colors)])
            ),
            row=1, col=2
        )

# Update layout
fig.update_xaxes(title_text="Aggressive Ratio", row=1, col=1)
fig.update_yaxes(title_text="Max Steps", row=1, col=1)
fig.update_xaxes(title_text="Aggressive Ratio", row=1, col=2)
fig.update_yaxes(title_text="Max Steps", row=1, col=2)

fig.update_layout(height=500, width=1400, showlegend=True, 
                  title_text="Aggressive Agents Impact on Evacuation Time by Scenario Type",
                  legend=dict(x=1.05, y=1))
fig.show()

# Summary statistics by scenario type
print("\n" + "=" * 80)
print("AGGRESSIVE AGENT IMPACT BY SCENARIO TYPE:")
print("=" * 80)

for scenario in scenario_types:
    scenario_max_steps = max_steps_agg_by_scenario[max_steps_agg_by_scenario['scenario_type'] == scenario]
    scenario_stats_detail = scenario_max_steps.groupby("aggressive_ratio")["max_steps"].agg([
        "count", "mean", "std", "min", "max"
    ]).round(2)
    
    print(f"\n{scenario}:")
    print(scenario_stats_detail)
    
    if len(scenario_stats_detail) > 0:
        baseline_scenario = scenario_stats_detail["mean"].iloc[0]
        max_scenario = scenario_stats_detail["mean"].max()
        print(f"  Baseline (0% aggressive): {baseline_scenario:.1f} steps")
        print(f"  Maximum ({scenario_stats_detail['mean'].idxmax()*100:.0f}% aggressive): {max_scenario:.1f} steps")
        print(f"  Impact: {((max_scenario - baseline_scenario) / baseline_scenario * 100):.1f}%")


Unique scenario types in data: ['MALL' 'SEATS']
Unique aggressive ratios: [np.float64(0.0), np.float64(0.1), np.float64(0.25), np.float64(0.5), np.float64(0.8), np.float64(1.0)]

Sample of data with scenario type:
    RunId scenario_type  aggressive_ratio  max_steps
0       0          MALL              1.00         29
1       1         SEATS              1.00         97
2       2          MALL              0.80         38
3       3         SEATS              0.80         92
4       4          MALL              0.50         33
5       5         SEATS              0.50        111
6       6          MALL              0.25         39
7       7         SEATS              0.25        125
8       8          MALL              0.10         37
9       9         SEATS              0.10        104
10     10          MALL              0.00         33
11     11         SEATS              0.00        101
12     12          MALL              1.00         38
13     13         SEATS              1.00   


AGGRESSIVE AGENT IMPACT BY SCENARIO TYPE:

MALL:
                  count   mean   std  min  max
aggressive_ratio                              
0.00               1000  35.25  3.66   27   52
0.10               1000  34.64  3.56   26   57
0.25               1000  34.12  3.65   25   51
0.50               1000  33.11  3.65   25   51
0.80               1000  32.35  3.76   24   57
1.00               1000  32.06  3.82   25   51
  Baseline (0% aggressive): 35.2 steps
  Maximum (0% aggressive): 35.2 steps
  Impact: 0.0%

SEATS:
                  count    mean    std  min  max
aggressive_ratio                                
0.00               1000  110.20  10.98   84  152
0.10               1000  107.15  10.59   80  158
0.25               1000  103.69  10.58   69  150
0.50               1000   99.59   9.93   74  151
0.80               1000   96.00   9.76   72  132
1.00               1000   94.42   9.52   71  136
  Baseline (0% aggressive): 110.2 steps
  Maximum (0% aggressive): 110.2 steps
  I

In [21]:

# Advanced visualization: Aggressive ratio impact by scenario_type AND initial_agents
import plotly.graph_objects as go

# Get max steps per run including scenario_type and initial_agents
max_steps_full = df_aggressive.groupby(["RunId", "scenario_type", "initial_agents", "aggressive_ratio"])["Step"].max().reset_index()
max_steps_full.columns = ["RunId", "scenario_type", "initial_agents", "aggressive_ratio", "max_steps"]

print("Data structure with all dimensions:")
print(max_steps_full.head(20))
print(f"\nUnique scenario types: {sorted(max_steps_full['scenario_type'].unique())}")
print(f"Unique initial_agents: {sorted(max_steps_full['initial_agents'].unique())}")
print(f"Unique aggressive ratios: {sorted(max_steps_full['aggressive_ratio'].unique())}")

# Create visualization with traces for each scenario_type + initial_agents combination
fig = go.Figure()

scenario_types = sorted(df_aggressive['scenario_type'].unique())
initial_agents_values = sorted(df_aggressive['initial_agents'].unique())
colors = ["blue", "red", "green", "orange", "purple", "brown", "pink", "cyan", "magenta", "lime"]
line_styles = ["solid", "dash", "dot", "dashdot"]

print(f"\nCreating traces for {len(scenario_types)} scenario types x {len(initial_agents_values)} agent counts")

trace_idx = 0
for scenario in scenario_types:
    for agent_count in initial_agents_values:
        # Filter data for this scenario + agent count combination
        combo_data = max_steps_full[
            (max_steps_full['scenario_type'] == scenario) & 
            (max_steps_full['initial_agents'] == agent_count)
        ]
        
        # Get stats per aggressive ratio
        combo_stats = combo_data.groupby("aggressive_ratio")["max_steps"].agg([
            "mean",
            "std",
        ]).reset_index()
        
        if len(combo_stats) > 0:
            color_idx = trace_idx % len(colors)
            dash_idx = (trace_idx // len(colors)) % len(line_styles)
            
            fig.add_trace(
                go.Scatter(
                    x=[f"{x*100:.0f}%" for x in combo_stats["aggressive_ratio"]],
                    y=combo_stats["mean"].values,
                    error_y=dict(type="data", array=combo_stats["std"].values, visible=True),
                    mode="lines+markers",
                    name=f"{scenario} - {agent_count} agents",
                    marker=dict(size=8),
                    line=dict(width=2, color=colors[color_idx], dash=line_styles[dash_idx])
                )
            )
            trace_idx += 1

# Update layout
fig.update_xaxes(title_text="Aggressive Ratio")
fig.update_yaxes(title_text="Mean Max Steps")
fig.update_layout(
    height=600, 
    width=1400, 
    showlegend=True,
    title_text="Aggressive Agents Impact: By Scenario Type and Initial Agent Count",
    legend=dict(x=1.02, y=1, xanchor='left', yanchor='top'),
    hovermode='x unified'
)
fig.show()

print(f"\n✓ Created {trace_idx} traces total")


Data structure with all dimensions:
    RunId scenario_type  initial_agents  aggressive_ratio  max_steps
0       0          MALL             125              1.00         29
1       1         SEATS             125              1.00         97
2       2          MALL             125              0.80         38
3       3         SEATS             125              0.80         92
4       4          MALL             125              0.50         33
5       5         SEATS             125              0.50        111
6       6          MALL             125              0.25         39
7       7         SEATS             125              0.25        125
8       8          MALL             125              0.10         37
9       9         SEATS             125              0.10        104
10     10          MALL             125              0.00         33
11     11         SEATS             125              0.00        101
12     12          MALL             125              1.00         3


✓ Created 2 traces total


In [22]:

# Detailed statistics table by scenario and agent count
print("\n" + "=" * 100)
print("DETAILED IMPACT ANALYSIS: AGGRESSIVE RATIO BY SCENARIO TYPE AND INITIAL AGENTS")
print("=" * 100)

for scenario in scenario_types:
    for agent_count in initial_agents_values:
        combo_data = max_steps_full[
            (max_steps_full['scenario_type'] == scenario) & 
            (max_steps_full['initial_agents'] == agent_count)
        ]
        
        combo_stats_detail = combo_data.groupby("aggressive_ratio")["max_steps"].agg([
            "count", "mean", "std", "min", "max"
        ]).round(2)
        
        if len(combo_stats_detail) > 0:
            print(f"\n{scenario} - {agent_count} agents:")
            print(combo_stats_detail)
            
            baseline = combo_stats_detail["mean"].iloc[0]
            max_val = combo_stats_detail["mean"].max()
            min_val = combo_stats_detail["mean"].min()
            max_ratio_idx = combo_stats_detail["mean"].idxmax()
            min_ratio_idx = combo_stats_detail["mean"].idxmin()
            
            print(f"  Baseline (0% aggressive): {baseline:.1f} steps")
            print(f"  Best ({min_ratio_idx*100:.0f}% aggressive): {min_val:.1f} steps ({((min_val - baseline) / baseline * 100):+.1f}%)")
            print(f"  Worst ({max_ratio_idx*100:.0f}% aggressive): {max_val:.1f} steps ({((max_val - baseline) / baseline * 100):+.1f}%)")



DETAILED IMPACT ANALYSIS: AGGRESSIVE RATIO BY SCENARIO TYPE AND INITIAL AGENTS

MALL - 125 agents:
                  count   mean   std  min  max
aggressive_ratio                              
0.00               1000  35.25  3.66   27   52
0.10               1000  34.64  3.56   26   57
0.25               1000  34.12  3.65   25   51
0.50               1000  33.11  3.65   25   51
0.80               1000  32.35  3.76   24   57
1.00               1000  32.06  3.82   25   51
  Baseline (0% aggressive): 35.2 steps
  Best (100% aggressive): 32.1 steps (-9.0%)
  Worst (0% aggressive): 35.2 steps (+0.0%)

SEATS - 125 agents:
                  count    mean    std  min  max
aggressive_ratio                                
0.00               1000  110.20  10.98   84  152
0.10               1000  107.15  10.59   80  158
0.25               1000  103.69  10.58   69  150
0.50               1000   99.59   9.93   74  151
0.80               1000   96.00   9.76   72  132
1.00               1000   94.42 

### Slow impact

### All Impact

### A* vs BFS

### Exit preference Impact

### Number of exits